In [1]:
from dotenv import load_dotenv
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
import numpy as np
from xgboost import XGBRegressor
import json
import os
import tempfile
from tqdm import tqdm
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import subprocess


In [2]:
import sys
from pathlib import Path
workspace_root = Path().cwd().parent.parent


print(f"Workspace root: {workspace_root}")
sys.path.insert(0, str(workspace_root))

from util import evaluate
from community_contributions.yeechern.modal_package import app, run_embedding, train_evaluate_xgboost

Workspace root: e:\Project\llm_engineering\week7


In [3]:
env_path = workspace_root.parent / ".env"
load_dotenv(override=True)

print("HF_TOKEN loaded!" if os.environ.get("HF_TOKEN") else "Failed to load HF_TOKEN")
print("MODAL_TOKEN_ID loaded!" if os.environ.get("MODAL_TOKEN_ID") else "Failed to load MODAL_TOKEN_ID")
print("MODAL_TOKEN_SECRET loaded!" if os.environ.get("MODAL_TOKEN_SECRET") else "Failed to load MODAL_TOKEN_SECRET")

HF_TOKEN loaded!
MODAL_TOKEN_ID loaded!
MODAL_TOKEN_SECRET loaded!


In [4]:
os.environ["MODAL_IMAGE_BUILDER_VERSION"] = "2025.06"

## Embedding on Modal.com

In [ ]:
DATASET_NAME = "ed-donner/items_prompts_full"

# get embedded dataset
embedding_model = "sentence-transformers/all-MiniLM-L12-v2"

with app.run():
    modal_volume_name, file_name = run_embedding.remote(DATASET_NAME, embedding_model = embedding_model, batch_size=512)

In [ ]:
local_data_dir = Path("./data")

subprocess.run([
    "modal", "volume", "get",
    modal_volume_name, file_name, str(local_data_dir)
], check=True)

In [15]:
dataset = load_from_disk(local_data_dir/ file_name)
dataset, dataset['train']["embedding"][0].shape if type(dataset['train']["embedding"][0]) == np.ndarray else len(dataset['train']["embedding"][0])

(DatasetDict({
     train: Dataset({
         features: ['prompt', 'completion', 'embedding'],
         num_rows: 800000
     })
     val: Dataset({
         features: ['prompt', 'completion', 'embedding'],
         num_rows: 10000
     })
     test: Dataset({
         features: ['prompt', 'completion', 'embedding'],
         num_rows: 10000
     })
 }),
 1024)

### Preprocessing

In [16]:
# price string to float
dataset = dataset.with_format("numpy")
dataset = dataset.map(lambda x : {'completion': float(x['completion'])}, batched=False)

Map:   0%|          | 0/800000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [17]:
# price normalization
scaler = StandardScaler()
scaler.fit(np.array(dataset['train']['completion']).reshape(-1, 1))

def normalize(batch):
    arr = np.asarray(batch["completion"], dtype=np.float64).reshape(-1, 1)
    return {"completion": scaler.transform(arr).flatten().tolist()}

train_dataset = dataset['train'].map(normalize, batched=True)
val_dataset = dataset['val'].map(normalize, batched=True)

Map:   0%|          | 0/800000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

## Train XGBoost model with embeddings

In [18]:
X_train = np.asarray(train_dataset['embedding'], dtype=np.float32)
y_train = train_dataset['completion']

X_val = np.asarray(val_dataset['embedding'], dtype=np.float32)
y_val = val_dataset['completion']

In [19]:
X_train

array([[-0.00746448,  0.00519005, -0.01778223, ...,  0.00985636,
        -0.04739556,  0.01257678],
       [-0.01077094,  0.01924235, -0.02182501, ...,  0.0222837 ,
        -0.03854798, -0.04548659],
       [-0.01994287,  0.04390248, -0.0449236 , ...,  0.01616319,
        -0.02944083, -0.01909604],
       ...,
       [-0.00998221, -0.02998138, -0.04090106, ...,  0.02066549,
        -0.01294392, -0.0103812 ],
       [-0.00243917, -0.00310332, -0.00777254, ..., -0.02933881,
        -0.00345797,  0.00931446],
       [-0.04873256, -0.00490532, -0.02472519, ...,  0.00353176,
        -0.00921897, -0.03223094]], shape=(800000, 1024), dtype=float32)

In [20]:
model = XGBRegressor(
    n_estimators=5000,
    max_depth=8,
    learning_rate=1e-1,
    early_stopping_rounds=10,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=True,
)

[0]	validation_0-rmse:0.94946
[1]	validation_0-rmse:0.93426
[2]	validation_0-rmse:0.92079
[3]	validation_0-rmse:0.90926
[4]	validation_0-rmse:0.89882
[5]	validation_0-rmse:0.88862
[6]	validation_0-rmse:0.87906
[7]	validation_0-rmse:0.87030
[8]	validation_0-rmse:0.86243
[9]	validation_0-rmse:0.85592
[10]	validation_0-rmse:0.84976
[11]	validation_0-rmse:0.84389
[12]	validation_0-rmse:0.83860
[13]	validation_0-rmse:0.83345
[14]	validation_0-rmse:0.82884
[15]	validation_0-rmse:0.82474
[16]	validation_0-rmse:0.82048
[17]	validation_0-rmse:0.81690
[18]	validation_0-rmse:0.81327
[19]	validation_0-rmse:0.80953
[20]	validation_0-rmse:0.80601
[21]	validation_0-rmse:0.80317
[22]	validation_0-rmse:0.80018
[23]	validation_0-rmse:0.79747
[24]	validation_0-rmse:0.79505
[25]	validation_0-rmse:0.79222
[26]	validation_0-rmse:0.78998
[27]	validation_0-rmse:0.78779
[28]	validation_0-rmse:0.78534
[29]	validation_0-rmse:0.78342
[30]	validation_0-rmse:0.78101
[31]	validation_0-rmse:0.77891
[32]	validation_0-

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,10
,enable_categorical,False
,eval_metric,None


In [21]:
def model_predict(item):
    inputs = item['embedding']
    
    prediction = model.predict(np.array([inputs], dtype=np.float32))
    return scaler.inverse_transform(np.array([prediction], dtype=np.float64)).flatten().tolist()[0]

evaluate(model_predict, dataset['test'])

### Try Remove Outliers from training data

In [24]:
q3 = np.percentile(train_dataset['completion'], 75)
q1 = np.percentile(train_dataset['completion'], 25)
cap = q3 + 1.5 * (q3 - q1)

train_dataset_capped = train_dataset.filter(lambda x: x['completion'] <= cap)

Filter:   0%|          | 0/800000 [00:00<?, ? examples/s]

In [ ]:
X_train = np.asarray(train_dataset_capped['embedding'], dtype=np.float32)
y_train = train_dataset_capped['completion']

model = XGBRegressor(
    n_estimators=5000,
    max_depth=8,
    learning_rate=1e-1,
    early_stopping_rounds=10,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=True,
)

[0]	validation_0-rmse:0.98154
[1]	validation_0-rmse:0.97265
[2]	validation_0-rmse:0.96497
[3]	validation_0-rmse:0.95774
[4]	validation_0-rmse:0.95087
[5]	validation_0-rmse:0.94417
[6]	validation_0-rmse:0.93856
[7]	validation_0-rmse:0.93335
[8]	validation_0-rmse:0.92860
[9]	validation_0-rmse:0.92424
[10]	validation_0-rmse:0.92032
[11]	validation_0-rmse:0.91647
[12]	validation_0-rmse:0.91269
[13]	validation_0-rmse:0.90958
[14]	validation_0-rmse:0.90660
[15]	validation_0-rmse:0.90346
[16]	validation_0-rmse:0.90083
[17]	validation_0-rmse:0.89825
[18]	validation_0-rmse:0.89558
[19]	validation_0-rmse:0.89330
[20]	validation_0-rmse:0.89134
[21]	validation_0-rmse:0.88936
[22]	validation_0-rmse:0.88746
[23]	validation_0-rmse:0.88549
[24]	validation_0-rmse:0.88371
[25]	validation_0-rmse:0.88213
[26]	validation_0-rmse:0.88046
[27]	validation_0-rmse:0.87883
[28]	validation_0-rmse:0.87728
[29]	validation_0-rmse:0.87575
[30]	validation_0-rmse:0.87438
[31]	validation_0-rmse:0.87309
[32]	validation_0-

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,10
,enable_categorical,False
,eval_metric,None


In [26]:
def model_predict(item):
    inputs = item['embedding']
    
    prediction = model.predict(np.array([inputs], dtype=np.float32))
    return scaler.inverse_transform(np.array([prediction], dtype=np.float64)).flatten().tolist()[0]

evaluate(model_predict, dataset['test'])

Disappointingly, the model performance is not improved after removing outliers.

## Try PCA to reduce the dimension of embeddings

In [27]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100)

X_train = np.asarray(train_dataset['embedding'], dtype=np.float32)
y_train = train_dataset['completion']

X_val = np.asarray(val_dataset['embedding'], dtype=np.float32)
y_val = val_dataset['completion']

X_train = pca.fit_transform(X_train)
X_val = pca.transform(X_val)

model = XGBRegressor(
    n_estimators=5000,
    max_depth=8,
    learning_rate=1e-1,
    early_stopping_rounds=10,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=True,
)

[0]	validation_0-rmse:0.94548
[1]	validation_0-rmse:0.92679
[2]	validation_0-rmse:0.91101
[3]	validation_0-rmse:0.89787
[4]	validation_0-rmse:0.88497
[5]	validation_0-rmse:0.87467
[6]	validation_0-rmse:0.86531
[7]	validation_0-rmse:0.85704
[8]	validation_0-rmse:0.85017
[9]	validation_0-rmse:0.84425
[10]	validation_0-rmse:0.83828
[11]	validation_0-rmse:0.83348
[12]	validation_0-rmse:0.82885
[13]	validation_0-rmse:0.82515
[14]	validation_0-rmse:0.82136
[15]	validation_0-rmse:0.81694
[16]	validation_0-rmse:0.81326
[17]	validation_0-rmse:0.81011
[18]	validation_0-rmse:0.80761
[19]	validation_0-rmse:0.80495
[20]	validation_0-rmse:0.80269
[21]	validation_0-rmse:0.80040
[22]	validation_0-rmse:0.79787
[23]	validation_0-rmse:0.79579
[24]	validation_0-rmse:0.79413
[25]	validation_0-rmse:0.79215
[26]	validation_0-rmse:0.79070
[27]	validation_0-rmse:0.78937
[28]	validation_0-rmse:0.78809
[29]	validation_0-rmse:0.78684
[30]	validation_0-rmse:0.78557
[31]	validation_0-rmse:0.78445
[32]	validation_0-

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,10
,enable_categorical,False
,eval_metric,None


In [28]:
def model_predict(item):
    inputs = item['embedding']
    inputs = pca.transform(np.array([inputs], dtype=np.float32))
    
    prediction = model.predict(inputs)
    return scaler.inverse_transform(np.array([prediction], dtype=np.float64)).flatten().tolist()[0]

evaluate(model_predict, dataset['test'])

Seem like PCA didn't improve the model performance either.

## Try using larger embedding model

In [ ]:
embedding_model = 'BAAI/bge-large-en-v1.5'

with app.run():
    modal_volume_name, file_name = run_embedding.remote(DATASET_NAME, embedding_model = embedding_model, batch_size=256)

In [1]:
from dotenv import load_dotenv
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
import numpy as np
from xgboost import XGBRegressor
import json
import os
import tempfile
from tqdm import tqdm
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import subprocess

import sys
from pathlib import Path
workspace_root = Path().cwd().parent.parent


print(f"Workspace root: {workspace_root}")
sys.path.insert(0, str(workspace_root))

from util import evaluate
from community_contributions.yeechern.modal_package import app, run_embedding, train_evaluate_xgboost
env_path = workspace_root.parent / ".env"
load_dotenv(override=True)

print("HF_TOKEN loaded!" if os.environ.get("HF_TOKEN") else "Failed to load HF_TOKEN")
print("MODAL_TOKEN_ID loaded!" if os.environ.get("MODAL_TOKEN_ID") else "Failed to load MODAL_TOKEN_ID")
print("MODAL_TOKEN_SECRET loaded!" if os.environ.get("MODAL_TOKEN_SECRET") else "Failed to load MODAL_TOKEN_SECRET")
os.environ["MODAL_IMAGE_BUILDER_VERSION"] = "2025.06"

Workspace root: e:\Project\llm_engineering\week7
HF_TOKEN loaded!
MODAL_TOKEN_ID loaded!
MODAL_TOKEN_SECRET loaded!


In [ ]:
modal_volume_name = "ed-donner-price-prediction"
DATASET_NAME = "ed-donner/items_prompts_full"
embedding_model = 'BAAI/bge-large-en-v1.5'

file_name = DATASET_NAME.split("/")[-1] + "_embedded_" + embedding_model.split("/")[-1]

with app.run():
    results = train_evaluate_xgboost.remote(file_name)
    

In [4]:
import pandas as pd
from util import Tester

results = pd.DataFrame(results)

In [5]:
results

,title,guess,truth,error,color
0,Excess V2 Distortion/Modulation Pedal,139.955819,219.00,79.044181,orange
1,Telpo Headlight Assembly for 2015‑2017 T...,198.578520,115.99,82.588520,red
2,NewPower99 6000 mAh Battery Replacement ...,-5.412011,54.95,60.362011,orange
3,EXMAX NG-10X Fresnel Lens Focusing Adapt...,97.654770,69.98,27.674770,green
4,Raven Pro Document Scanner Stand,205.466645,29.85,175.616645,red
...,...,...,...,...,...
195,MonMount Triple Monitor Freestanding Tri...,161.825647,79.99,81.835647,red
196,"Xiaomi Redmi 10 2022 6.5"" 50MP Quad Came...",157.883976,169.99,12.106024,green
197,Volvo AC Compressor & Clutch Set for C70...,286.177941,280.41,5.767941,green
198,Refresh 3‑Pack NSF‑53 Refrigerator Water...,6.023785,50.99,44.966215,orange


In [11]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   title   200 non-null    object 
 1   guess   200 non-null    float64
 2   truth   200 non-null    float64
 3   error   200 non-null    float64
 4   color   200 non-null    object 
dtypes: float64(3), object(2)
memory usage: 7.9+ KB


In [12]:
tester = Tester(model_predict,None,None,len(results))
tester.titles = results['title'].tolist()
tester.guesses = results['guess'].tolist()
tester.truths = results['truth'].tolist()
tester.errors = results['error'].tolist()
tester.colors = results['color'].tolist()

tester.report()

## Conclusion

- Remove outliers and PCA didn't improve the model performance.
- BAAI/bge-large-en-v1.5 embedding didn't superior to sentence-transformers/all-MiniLM-L12-v2 embedding (I have also tried sentence-transformers/all-MiniLM-L6-v2, which is smaller than L12-v2, but the performance is slightly worse than L12-v2).

## Future Work

- Try hyperparameter tuning with BAAI/bge-large-en-v1.5, since BAAI/bge-large-en-v1.5 embedded dataset requires more memory, modal.com is my only option to train xgboost, and I have not tried hyperparameter tuning on modal.com yet.